# Trustworthy AI Pipeline (Advanced Version with Explainability & Selective Prediction)

This notebook implements a research-grade, production-ready machine learning pipeline for Human Activity Recognition (HAR).

### 📌 Pipeline Flow
1. **Input arrives**
2. **Drift monitoring (parallel)**: Drift monitoring runs in parallel to prediction.
3. **Model prediction**: Fast primary model (Logistic Regression).
4. **Calibration / confidence evaluation**: Statistical assessment of prediction confidence.
5. **Selective decision**: 3-tier dynamic prediction (Accept / Defer / Reject).
6. **Explainability checks**: Consistency verification using Explanation Stability Score (ESS).

*Note: We separate prediction confidence from trust using calibration, data stability, and explanation consistency.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import shap
from scipy.stats import ks_2samp

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")
print("\u2705 Libraries imported successfully.")

✅ Libraries imported successfully.


## 1. Input Data
Loading the Human Activity Recognition (HAR) dataset.

In [2]:
print("\u23f3 Fetching dataset (this may take a moment)...")
try:
    # Try fetching HAR from OpenML
    dataset = fetch_openml(data_id=1478, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target
    print("\u2705 Real HAR dataset loaded.")
except Exception as e:
    print("\u26a0\ufe0f Could not fetch HAR dataset. Falling back to synthetic HAR-like data for demonstration.")
    from sklearn.datasets import make_classification
    X_arr, y_arr = make_classification(n_samples=5000, n_features=561, n_informative=100, n_classes=6, random_state=42)
    X = pd.DataFrame(X_arr, columns=[f"Feature_{i}" for i in range(561)])
    y = pd.Series(y_arr).astype(str)

# Stratified Splitting
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_cal = pd.DataFrame(scaler.transform(X_cal), columns=X.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_cal_enc = le.transform(y_cal)
y_test_enc = le.transform(y_test)

print(f"\ud83d\udcca Training shape: {X_train.shape}")
print(f"\ud83d\udcca Calibration shape: {X_cal.shape}")
print(f"\ud83d\udcca Evaluation shape: {X_test.shape}")

⏳ Fetching dataset (this may take a moment)...


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/jupyter_client/session.py", line 95, in json_packer
    return json.dumps(
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 56-57: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/zmq/eventloop/zmqstream.py", line 550, in _run_callback
    f = callback(*args, **kwargs)
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/ipykernel/iostream.py", line 170, in _handle_event
    event_f()
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/ipykernel/iostream.py", line 649, in _flush
    self.session.send(
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/jupyter_client/session.py", line 852, in send
    

## 2. Parallel Drift Monitoring (Input Drift)
Drift monitoring runs in parallel to prediction. We use KS-Test and PSI to monitor continuous distribution shifts.

In [3]:
def calculate_psi(expected, actual, bins=10):
    """Calculate Population Stability Index (PSI)"""
    expected_binned, bin_edges = np.histogram(expected, bins=bins, density=True)
    actual_binned, _ = np.histogram(actual, bins=bin_edges, density=True)
    
    # Avoid zero division
    expected_binned = np.where(expected_binned == 0, 1e-6, expected_binned)
    actual_binned = np.where(actual_binned == 0, 1e-6, actual_binned)
    
    # Normalize
    expected_binned = expected_binned / sum(expected_binned)
    actual_binned = actual_binned / sum(actual_binned)
    
    psi_value = np.sum((actual_binned - expected_binned) * np.log(actual_binned / expected_binned))
    return psi_value

def monitor_drift(X_ref, X_curr, threshold_psi=0.2):
    print("\ud83d\udd0d Running Parallel Drift Monitor...")
    drifted_features = 0
    for col in X_ref.columns[:50]: # Check top 50 for speed
        psi_val = calculate_psi(X_ref[col], X_curr[col])
        if psi_val > threshold_psi:
            drifted_features += 1
    
    pct_drift = (drifted_features / 50) * 100
    print(f"\u2705 Drift Check Complete: {pct_drift}% of monitored features shifted.")
    return pct_drift

_ = monitor_drift(X_train, X_test)

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/jupyter_client/session.py", line 95, in json_packer
    return json.dumps(
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/zmq/eventloop/zmqstream.py", line 550, in _run_callback
    f = callback(*args, **kwargs)
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/ipykernel/iostream.py", line 170, in _handle_event
    event_f()
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/ipykernel/iostream.py", line 649, in _flush
    self.session.send(
  File "/Users/aaditgupta/Library/Python/3.9/lib/python/site-packages/jupyter_client/session.py", line 852, in send
    

## 3. Model Prediction & Calibration
We train three models:
1. **Logistic Regression (Primary)**: Fast and well-calibrated via log-loss.
2. **XGBoost (Fallback)**: Highly robust but requires post-hoc isotonic calibration.
3. **PyTorch MLP**: Deep learning baseline.

*Note: Neural networks tend to be overconfident due to poor calibration and overfitting.*

In [ ]:
print("\u23f3 Training Primary Model (Logistic Regression)...")
lr_model = LogisticRegression(max_iter=200, random_state=42, multi_class='multinomial')
lr_model.fit(X_train, y_train_enc)

print("\u23f3 Training Fallback Model (XGBoost) and Calibrating...")
xgb_base = xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=42, eval_metric='mlogloss')
xgb_base.fit(X_train, y_train_enc)
xgb_calibrated = CalibratedClassifierCV(xgb_base, method='isotonic', cv='prefit')
xgb_calibrated.fit(X_cal, y_cal_enc)

print("\u23f3 Training PyTorch MLP...")
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.net(x)

mlp = SimpleMLP(X_train.shape[1], len(le.classes_))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=0.005)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_enc, dtype=torch.long)
dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

mlp.train()
for epoch in range(5):
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        loss = criterion(mlp(batch_X), batch_y)
        loss.backward()
        optimizer.step()
mlp.eval()

def predict_proba_mlp(X):
    with torch.no_grad():
        t = torch.tensor(X.values, dtype=torch.float32)
        return torch.softmax(mlp(t), dim=1).numpy()

print("\u2705 All models trained.")

⏳ Training Primary Model (Logistic Regression)...
⏳ Training Fallback Model (XGBoost) and Calibrating...
⏳ Training PyTorch MLP...


## 4. Confidence Evaluation & Dynamic Thresholds
We compute ECE and TPS, then determine dynamic acceptance and rejection thresholds.

*Note: Percentiles define relative ranking of predictions, while calibration ensures that confidence values are meaningful in absolute terms.*

In [ ]:
def compute_ece(y_true, y_proba_mat, n_bins=10):
    confidence = np.max(y_proba_mat, axis=1)
    predicted = np.argmax(y_proba_mat, axis=1)
    correct = (predicted == y_true)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.
    for b in range(n_bins):
        mask = (confidence >= bin_edges[b]) & (confidence < bin_edges[b+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(correct[mask].mean() - confidence[mask].mean())
    return ece / len(y_true)

def compute_tps(coverage, sel_acc, ece):
    return (coverage * sel_acc) / (1 + ece)

# Dynamic Thresholds Calculation
lr_test_proba = lr_model.predict_proba(X_test)
lr_test_conf = np.max(lr_test_proba, axis=1)

tau_high = np.percentile(lr_test_conf, 20)
tau_low = np.percentile(lr_test_conf, 5)

# We enforce \u03c4_high > \u03c4_low to preserve a valid decision boundary.
if tau_low >= tau_high:
    tau_high = min(1.0, tau_low + 1e-5)

print(f"\ud83d\udcca Dynamic Thresholds Computed:")
print(f"   \u03c4_high (20th percentile): {tau_high:.4f}")
print(f"   \u03c4_low  (5th percentile):  {tau_low:.4f}")

## 5. Explainability & ESS
We compute global SHAP values and define the Explanation Stability Score (ESS).

*Note: ESS is a heuristic and may be affected by correlated features, so it should be treated as a heuristic signal rather than a strict guarantee of explanation validity.*

In [ ]:
print("\u23f3 Computing SHAP values for XGBoost Fallback...")
# Use a background subset for speed
X_bg = X_train.sample(100, random_state=42)
explainer = shap.TreeExplainer(xgb_base)
shap_global = explainer(X_bg)

# Global importance: mean absolute SHAP value across samples for each class
global_importance = np.abs(shap_global.values).mean(axis=0) # shape: (Features, Classes)

def calculate_ess(local_shap, predicted_class, global_imp_matrix):
    """
    ESS measures consistency of reasoning. 
    Low ESS -> unstable explanation -> reject.
    """
    local_class_shap = np.abs(local_shap[:, predicted_class])
    global_class_shap = global_imp_matrix[:, predicted_class]
    
    k = 5
    # Get indices of top k features
    top_local = set(np.argsort(-local_class_shap)[:k])
    top_global = set(np.argsort(-global_class_shap)[:k])
    
    intersection = top_local.intersection(top_global)
    ess = len(intersection) / k
    return ess

# Display Global Summary Plot
plt.figure(figsize=(8, 4))
shap.summary_plot(shap_global.values[:,:,0], X_bg, plot_type="bar", max_display=10, show=False)
plt.title("Global SHAP Feature Importance (Class 0)")
plt.show()

## 6. Selective Prediction Engine
3-tier system:
* **ACCEPT** \u2192 high confidence
* **DEFER** \u2192 fallback model (XGBoost)
* **REJECT** \u2192 low confidence or low ESS

In [ ]:
def selective_pipeline(X, y_true):
    n = len(X)
    lr_proba = lr_model.predict_proba(X)
    lr_conf = np.max(lr_proba, axis=1)
    lr_pred = np.argmax(lr_proba, axis=1)
    
    xgb_proba = xgb_calibrated.predict_proba(X)
    xgb_pred = np.argmax(xgb_proba, axis=1)
    
    shap_local = explainer(X)
    
    decision = np.empty(n, dtype=object)
    final_pred = np.empty(n, dtype=int)
    
    for i in range(n):
        if lr_conf[i] >= tau_high:
            decision[i] = 'ACCEPT'
            final_pred[i] = lr_pred[i]
        elif lr_conf[i] >= tau_low:
            # Check Explanation Stability Score (ESS)
            ess = calculate_ess(shap_local.values[i], xgb_pred[i], global_importance)
            if ess < 0.4: # Less than 40% overlap
                decision[i] = 'REJECT_ESS'
                final_pred[i] = -1
            else:
                decision[i] = 'DEFER'
                final_pred[i] = xgb_pred[i]
        else:
            decision[i] = 'REJECT_CONF'
            final_pred[i] = -1
            
    # Metrics
    accepted_mask = (decision == 'ACCEPT') | (decision == 'DEFER')
    coverage = accepted_mask.sum() / n
    sel_acc = accuracy_score(y_true[accepted_mask], final_pred[accepted_mask]) if coverage > 0 else 0
    
    # ECE for selective subset
    conf_covered = np.where(decision == 'ACCEPT', lr_conf, np.where(decision == 'DEFER', np.max(xgb_proba, axis=1), 0.0))
    ece_sel = 0.0
    if coverage > 0:
        bin_edges = np.linspace(0, 1, 11)
        correct_cov = (final_pred[accepted_mask] == y_true[accepted_mask])
        conf_cov_vals = conf_covered[accepted_mask]
        for b in range(10):
            mask = (conf_cov_vals >= bin_edges[b]) & (conf_cov_vals < bin_edges[b+1])
            if mask.sum() > 0:
                ece_sel += mask.sum() * abs(correct_cov[mask].mean() - conf_cov_vals[mask].mean())
        ece_sel /= accepted_mask.sum()
        
    tps = compute_tps(coverage, sel_acc, ece_sel)
    
    return {
        "decisions": pd.Series(decision).value_counts().to_dict(),
        "coverage": coverage,
        "sel_acc": sel_acc,
        "ece": ece_sel,
        "tps": tps,
        "final_pred": final_pred,
        "decision_array": decision,
        "shap_local": shap_local
    }

print("\u2705 Selective Engine Ready.")

## 7. Clean Results Section & Error Analysis
Running the pipeline and comparing against baseline models.

In [ ]:
# 1. Baselines (100% Coverage)
lr_pred_base = np.argmax(lr_model.predict_proba(X_test), axis=1)
xgb_pred_base = np.argmax(xgb_calibrated.predict_proba(X_test), axis=1)
mlp_pred_base = np.argmax(predict_proba_mlp(X_test), axis=1)

lr_ece = compute_ece(y_test_enc, lr_model.predict_proba(X_test))
xgb_ece = compute_ece(y_test_enc, xgb_calibrated.predict_proba(X_test))
mlp_ece = compute_ece(y_test_enc, predict_proba_mlp(X_test))

lr_tps = compute_tps(1.0, accuracy_score(y_test_enc, lr_pred_base), lr_ece)
xgb_tps = compute_tps(1.0, accuracy_score(y_test_enc, xgb_pred_base), xgb_ece)
mlp_tps = compute_tps(1.0, accuracy_score(y_test_enc, mlp_pred_base), mlp_ece)

# 2. Selective Engine
res = selective_pipeline(X_test, y_test_enc)

# Display Comparison
results_df = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost", "PyTorch MLP", "Selective AI Pipeline"],
    "Coverage": ["100.00%", "100.00%", "100.00%", f"{res['coverage']*100:.2f}%"],
    "Accuracy": [
        f"{accuracy_score(y_test_enc, lr_pred_base)*100:.2f}%",
        f"{accuracy_score(y_test_enc, xgb_pred_base)*100:.2f}%",
        f"{accuracy_score(y_test_enc, mlp_pred_base)*100:.2f}%",
        f"{res['sel_acc']*100:.2f}%"
    ],
    "ECE": [f"{lr_ece:.4f}", f"{xgb_ece:.4f}", f"{mlp_ece:.4f}", f"{res['ece']:.4f}"],
    "TPS": [f"{lr_tps:.4f}", f"{xgb_tps:.4f}", f"{mlp_tps:.4f}", f"{res['tps']:.4f}"]
})

print("\ud83d\udcca EVALUATION RESULTS:")
import IPython.display as display
display.display(results_df)

print("\n\u2699\ufe0f PIPELINE DECISION BREAKDOWN:")
for k, v in res['decisions'].items():
    print(f"   {k}: {v} samples")

### Local Error Analysis (SHAP)
Examining a correct vs misclassified sample.

In [ ]:
decision_arr = res['decision_array']
final_pred = res['final_pred']
shap_local = res['shap_local']

accepted_mask = (decision_arr == 'ACCEPT') | (decision_arr == 'DEFER')
correct_mask = (final_pred == y_test_enc) & accepted_mask
wrong_mask = (final_pred != y_test_enc) & accepted_mask

correct_idx = np.where(correct_mask)[0][0] if correct_mask.sum() > 0 else None
wrong_idx = np.where(wrong_mask)[0][0] if wrong_mask.sum() > 0 else None

if correct_idx is not None:
    print(f"\n\u2705 CORRECT PREDICTION (Sample {correct_idx}) - True: {y_test_enc[correct_idx]}, Pred: {final_pred[correct_idx]}")
    plt.figure(figsize=(8, 3))
    shap.plots.waterfall(shap_local[correct_idx, :, final_pred[correct_idx]], show=False, max_display=5)
    plt.title("Correct Decision SHAP")
    plt.show()

if wrong_idx is not None:
    print(f"\n\u274c FAILED PREDICTION (Sample {wrong_idx}) - True: {y_test_enc[wrong_idx]}, Pred: {final_pred[wrong_idx]}")
    plt.figure(figsize=(8, 3))
    shap.plots.waterfall(shap_local[wrong_idx, :, final_pred[wrong_idx]], show=False, max_display=5)
    plt.title("Failed Decision SHAP")
    plt.show()
else:
    print("\n\ud83c\udf89 No misclassified samples passed the selective engine!")

## 8. Drift + Explanation Drift
Simulating sensor noise to observe its impact on explanation drift.

*Note: Even if the input looks stable, explanation drift can indicate reasoning change.*

In [ ]:
print("\ud83d\udca5 Injecting Gaussian Noise (Sensor Failure Simulation)...")
noise_level = 0.5
X_noisy = X_test + np.random.normal(0, noise_level, X_test.shape)

# Monitor Input Drift
print("\n\ud83d\udd0d Checking Input Drift:")
drift_pct = monitor_drift(X_train, X_noisy)

# Run Selective Pipeline on Noisy Data
res_noisy = selective_pipeline(X_noisy, y_test_enc)

print(f"\n\ud83d\udcca NOISY DATA RESULTS:")
print(f"   Coverage: {res_noisy['coverage']*100:.2f}%")
print(f"   Selective Accuracy: {res_noisy['sel_acc']*100:.2f}%")
print(f"   TPS: {res_noisy['tps']:.4f}")

print("\n\u2699\ufe0f NOISY DATA DECISION BREAKDOWN:")
for k, v in res_noisy['decisions'].items():
    print(f"   {k}: {v} samples")
    
if 'REJECT_ESS' in res_noisy['decisions']:
    print("\n\u26a0\ufe0f Explanation Drift Detected: The model rejected samples specifically because the SHAP explanations (reasoning) diverged from global expectations, protecting the system from confidently wrong predictions.")